# Julia essentials: enough to drive a climate simulation

*Arrays, broadcasting, multiple dispatch, and the language you need to read a model script.*

You just watched three simulations that are, underneath, short Julia scripts. This notebook gives you
exactly the Julia you need to read and write those scripts — no more: enough to make the configuration
files of the rest of the week *readable*, whether they set up an ocean, an atmosphere, or coupled sea
ice, and to change them with confidence. Everything runs on your laptop.

We lean on a handful of ideas, in order: **arrays and broadcasting** (how fields are stored and updated),
**functions** and **multiple dispatch** (why independent pieces compose), and a closing **diffusion step**
that ties them together — which we then watch drop down to the machine code the compiler emits.

## Installing and running Julia

Install Julia with [`juliaup`](https://github.com/JuliaLang/juliaup), the official version manager:

- **macOS / Linux:** `curl -fsSL https://install.julialang.org | sh`
- **Windows:** `winget install julia -s msstore` (or the installer at [julialang.org](https://julialang.org/downloads/))

`juliaup` keeps Julia current (`juliaup update`) and can hold several versions side by side. Typing `julia`
opens the **REPL**, the interactive prompt where everything below also runs. It has a few modes, each one
key away at an empty prompt:

| key | prompt | what it does |
|---|---|---|
| (default) | `julia>` | run Julia code |
| `]` | `pkg>` | the **package manager** (add / remove / update) |
| `?` | `help?>` | documentation for any function or type |
| `;` | `shell>` | a one-off shell command |

You can also run this material as a **Jupyter** notebook (via `IJulia`), a **Pluto** notebook, or in **VS
Code** with the Julia extension — the code is identical in all of them.

## Packages and environments

Code lives in *packages*, and every project carries its own *environment* — a `Project.toml` of direct
dependencies and a `Manifest.toml` pinning exact versions, reproducible to the digit. `Pkg` drives it: the
`]` mode in the REPL, or the `Pkg.` functions from code. Activate this notebook's environment and look inside:

In [ ]:
using Pkg
Pkg.activate("..")        # the notebook is in tutorials/day1/notebooks, so ".." is tutorials/day1
Pkg.status()              # the direct dependencies of this environment

On a fresh machine you would first run `Pkg.instantiate()` to download the exact versions the
`Manifest.toml` pins — this is how an environment is reproduced. Here it is already instantiated, so it
returns at once:

In [ ]:
Pkg.instantiate()         # install whatever the Manifest pins that isn't present yet (a no-op here)

Adding a dependency is `Pkg.add("PackageName")` (or `]add PackageName` in the REPL). To show it
*without* changing this environment, we add a tiny package inside a throwaway temporary environment, then
switch back:

In [ ]:
Pkg.activate(; temp = true)   # a disposable environment, so tutorials/day1 is untouched
Pkg.add("Example")            # the canonical tiny demo package
Pkg.status()
Pkg.activate("..")            # back to the day1 environment for the rest of the notebook

Back in the day1 environment, `using` brings package names into scope. When you need more, the docs are
excellent — the [Julia manual](https://docs.julialang.org), [Discourse](https://discourse.julialang.org),
and each package's own runnable examples. We keep the imports here modest:

In [1]:
using Statistics          # mean, std — small conveniences from the standard library
using Printf              # @sprintf, for clean numeric output
using InteractiveUtils    # @code_llvm, @code_native — to peek at the compiled code

## Variables, types, and a word on speed

Variables are dynamically typed, but every value has a concrete type, and that type is *why* Julia is
fast: the compiler specializes each function on the types it is called with, emitting machine code as
tight as C for that particular combination of arguments. You rarely write a type annotation — values
carry their type, and the compiler does the rest.

In [35]:
temperature = 4.0
salinity    = 35.0
wind_speed  = 1.f0
wind_name   = "zonal_velocity"
typeof(temperature), typeof(salinity), typeof(wind_speed), typeof(wind_name)

(Float64, Float64, Float32, String)

### Types form a tree

Concrete types like `Float64` are the leaves of an *abstract* hierarchy. The temperature we just stored is
a `Float64`, but it is therefore also a `Real`, an `AbstractFloat`, and a `Number` — `<:` ("is a subtype
of") reads that relationship off directly. This tree is the backbone of *multiple dispatch* further on: a
method written for any *supertype* automatically accepts every concrete type beneath it.

In [ ]:
supertype(Float64)        # AbstractFloat — one step up the tree
temperature isa Real      # true: our temperature is a Real, so anything written for Real accepts it
subtypes(AbstractFloat)   # the concrete floating-point leaves (Float16, Float32, Float64, …)

## Under the hood: high-level, yet you can read the machine code

That type-driven specialization is not a black box. `@code_llvm` prints the LLVM intermediate representation the
compiler emits — and because it specializes on the argument types, the *same* source becomes *different*
instructions for a `Float64` and an `Int`:

In [3]:
square(x) = x * x
@code_llvm debuginfo=:none square(2.0)     # a Float64 → floating-point multiply: `fmul double`

; Function Signature: square(Float64)
define double @julia_square_1915(double %"x::Float64") #0 {
top:
  %0 = fmul double %"x::Float64", %"x::Float64"
  ret double %0
}


Call it on an integer and the compiler emits integer arithmetic instead — one function, two machine codes:

In [5]:
@code_llvm debuginfo=:none square(2)       # an Int → `mul i64`

; Function Signature: square(Int64)
define i64 @julia_square_1978(i64 signext %"x::Int64") #0 {
top:
  %0 = mul i64 %"x::Int64", %"x::Int64"
  ret i64 %0
}


`@code_native` drops one level further, to the CPU's own assembly: a single hardware multiply, exactly
what a C compiler would emit for the same operation.

In [6]:
@code_native debuginfo=:none square(2.0)

	.section	__TEXT,__text,regular,pure_instructions
	.build_version macos, 16, 0
	.globl	_julia_square_2029              ; -- Begin function julia_square_2029
	.p2align	2
_julia_square_2029:                     ; @julia_square_2029
; Function Signature: square(Float64)
; %bb.0:                                ; %top
	;DEBUG_VALUE: square:x <- $d0
	;DEBUG_VALUE: square:x <- $d0
	fmul	d0, d0, d0
	ret
                                        ; -- End function
	.section	__DATA,__const
	.p2align	3, 0x0                          ; @"+Core.Float64#2031"
"l_+Core.Float64#2031":
	.quad	"l_+Core.Float64#2031.jit"

.set "l_+Core.Float64#2031.jit", 4863090832
.subsections_via_symbols


So Julia is high-level when you want it — you wrote no types — and low-level when you need it: you can
read exactly what runs. The same compiler turns the high-level array code below into the tight machine
code that makes Julia fast.

## Arrays — how a field is stored

An ocean field is an array. Build them explicitly, or from a function over a range:

In [7]:
depths = [0.0, -10.0, -50.0, -200.0, -1000.0]      # a 1D array, a column of interfaces
field  = zeros(4, 4)                               # a 4×4 array of zeros, ready to fill
size(field), length(depths)

((4, 4), 5)

Indexing is 1-based and the first index is fastest-varying (column-major), exactly like Fortran — so
the loops you may know from NEMO, MITgcm, or an atmosphere dycore keep the same memory-friendly order here:

In [8]:
depths[1], depths[end]                             # first and last; `end` is the last index

(0.0, -1000.0)

Ranges are lazy and cheap; `collect` materializes one if you ever need to:

In [9]:
x = range(0, 2π, length = 128)                     # 128 points around a periodic domain
Δx = step(x)

0.049473900056532176

## Tuples and named tuples — how settings travel together

A *tuple* is a fixed, ordered group of values, written with parentheses. Grid sizes and domain extents
ride around the model as tuples — `size = (128, 128, 64)`, `extent = (Lx, Ly, Lz)` — and you index or
destructure them like a tiny, immutable array:

In [10]:
grid_size = (128, 128, 64)                          # (Nx, Ny, Nz)
Nx, Ny, Nz = grid_size                              # destructuring, in one line
grid_size[3], Nx * Ny * Nz                          # index by position; total number of cells

(64, 1048576)

A *named tuple* labels each field, so a bundle of settings reads like the keyword arguments it usually
becomes. Access is by name, and the names are part of the type — immutable and allocation-light:

In [11]:
surface = (T = 4.0, S = 35.0, depth = 0.0)          # a labelled record of surface conditions
surface.T, surface.S                                # reach in by name

(4.0, 35.0)

## Control flow, loops, and comprehensions

The usual `if`/`elseif`/`else` and `for`/`while` are all here. A short conditional classifies a water
column by the sign of its buoyancy frequency `N²`:

In [12]:
stratification(N²) = N² > 0 ? "stable" : N² < 0 ? "unstable" : "neutral"   # the ternary `cond ? a : b`
stratification(1e-5), stratification(-1e-5)

("stable", "unstable")

`for` iterates anything iterable; mutating an array inside the loop needs no `global`, so building a
column interface-by-interface is natural:

In [13]:
column = Float64[]
for d in (10.0, 50.0, 200.0, 1000.0)
    push!(column, -d)                               # grow the vector in place
end
column

4-element Vector{Float64}:
   -10.0
   -50.0
  -200.0
 -1000.0

A *comprehension* does the same in one expression — the compact way to lay down a coordinate or a
stretched vertical grid, finer near the surface:

In [14]:
z_faces = [-1000.0 * (1 - i / 8)^2 for i in 0:8]    # 9 interfaces spanning −1000 m to the surface
length(z_faces), z_faces[begin], z_faces[end]

(9, -1000.0, -0.0)

## Broadcasting — the central idiom

A trailing dot applies any scalar operation element-wise, and *fuses* adjacent dotted operations into a
single pass with no temporaries. This is how you will write every field update for the rest of the week:

In [15]:
c = @. sin(x)                                      # @. dots the whole expression: c[i] = sin(x[i])
c² = c .^ 2                                         # element-wise square
rms = sqrt(mean(c²))                               # reductions read like maths
@sprintf("rms = %.4f", rms)

"rms = 0.7043"

The fused form `@. a + b * c` allocates *one* array, not three — the same code, the same performance
story, whether `a, b, c` live on the CPU or a GPU.

## Functions

Short functions get a one-line form; longer ones a block. A trailing `!` is a convention — not syntax —
marking a function that *mutates* its first argument, the workhorse pattern for updating fields in place:

In [16]:
buoyancy(T, S; g = 9.81, α = 2e-4, β = 8e-4) = g * (α * T - β * S)   # one-line, with keyword arguments

function normalize!(field)                          # the ! signals: this overwrites `field`
    field .-= mean(field)                           # broadcasting again, in place
    field ./= std(field)
    return field
end

normalize!(copy(c))                                 # operate on a copy so `c` survives for later cells
buoyancy(4.0, 35.0)

-0.266832

## Anonymous functions

A function does not need a name. The arrow form `(args) -> body` makes one on the spot — exactly how you
hand an *initial condition* or a *forcing* to the model, as a function of space:

In [17]:
initial_temperature = (x, y, z) -> 4.0 + 0.01z      # warm at the surface, cooling with depth
initial_temperature(0, 0, -100)

3.0

They are most common as an argument to another function. `map` applies one to every element and `filter`
keeps the ones that pass a test — each takes a small anonymous function as its first argument:

In [18]:
map(z -> initial_temperature(0, 0, z), z_faces)     # the temperature profile on our stretched grid

filter(d -> d < -100, column)                       # keep only the interfaces below 100 m

2-element Vector{Float64}:
  -200.0
 -1000.0

## Multiple dispatch — why the stack composes

A function is one *operation*; its *methods* are concrete implementations chosen by the types of the
arguments. New types slot into existing functions, so independent packages cooperate without ever
having been written for one another. A tiny ocean-flavored example:

In [19]:
abstract type Stratification end
struct Stable   <: Stratification end
struct Unstable <: Stratification end

mixing(::Stable)   = "weak diffusive mixing"
mixing(::Unstable) = "vigorous convective mixing"

mixing(Stable()), mixing(Unstable())

("weak diffusive mixing", "vigorous convective mixing")

Adding a *third* regime is a new type plus one method — the existing `mixing` calls, and any code built
on top of them, are untouched:

In [20]:
struct Neutral <: Stratification end
mixing(::Neutral) = "mechanical mixing only"
mixing(Neutral())

"mechanical mixing only"

Dispatch chooses on *all* the arguments at once — something single-dispatch object systems cannot do.
In Python a method belongs to one object, so `a.encounter(b)` dispatches on the type of `a` alone, and
the second type becomes a manual `isinstance` ladder you have to maintain by hand:

```python
class WarmWater:
    def encounter(self, other):            # keyed to WarmWater (self), blind to `other`
        if isinstance(other, SeaIce):
            return "basal melt"
        elif isinstance(other, ColdAir):
            return "heat loss, then convection"
        raise NotImplementedError          # and every new pair edits this one method
```

Julia keys the behaviour to *both* types — one method per pair, no ladder. We can even group the types
and their interactions into a *module*, the unit Julia uses to namespace a package; `export` makes the
chosen names available once we bring the module into scope with `using`:

In [21]:
module AirSeaExchange
    export WarmWater, SeaIce, ColdAir, encounter

    struct WarmWater end
    struct SeaIce    end
    struct ColdAir   end

    encounter(::WarmWater, ::SeaIce)    = "basal melt"
    encounter(::ColdAir,   ::SeaIce)    = "ice growth"
    encounter(::ColdAir,   ::WarmWater) = "heat loss, then convection"
    encounter(::T, ::T) where {T}       = "no exchange — same medium on both sides"   # one method, all (X, X)
    encounter(a, b)                     = encounter(b, a)                              # commutative: each pair once
end

using .AirSeaExchange

The `where {T}` method matches any medium paired with *itself*, and the last method makes `encounter`
commutative — so the three named pairs cover both orderings, and like media need no special case:

In [22]:
encounter(WarmWater(), SeaIce()), encounter(SeaIce(), WarmWater()), encounter(ColdAir(), ColdAir())

("basal melt", "basal melt", "no exchange — same medium on both sides")

Extending the model is a new type and a single method, written from *outside* the module and touching
nothing that already exists — a calving glacier that meets warm water, in either order:

In [23]:
struct Glacier end
AirSeaExchange.encounter(::Glacier, ::WarmWater) = "frontal melt and calving"

encounter(Glacier(), WarmWater()), encounter(WarmWater(), Glacier())

("frontal melt and calving", "frontal melt and calving")

A *function* is one name; its *methods* are the implementations dispatch chooses between. After all those
definitions, `encounter` is a single function carrying several methods — `methods` lists them:

In [ ]:
methods(encounter)        # one function, several methods — one per air/sea/ice pair, plus the fallbacks

This is exactly the mechanism behind the showcase: one `advection` operator serving every field type, one
model assembled from independently-written ocean, ice, and atmosphere components — each free to add its
own types and methods to functions it does not own.

## Composability: code that never heard of your types

The real dividend of dispatch is that a function written for plain numbers keeps working on types it
was never designed for. `Measurements.jl` supplies a number that carries an uncertainty, written `a ± b`;
our own `buoyancy`, unchanged, propagates that uncertainty through every operation inside it:

In [ ]:
using Measurements

buoyancy(4.0 ± 0.5, 35.0 ± 0.1)            # error bars in, error bars out — and we wrote no special code

Neither we nor the authors of `Measurements` arranged this — it is multiple dispatch composing two
independent pieces of code, the property the whole stack is built on. We will watch it survive a *whole
field update*, too, once we have built one — just below.

## Putting it together: one diffusion step

Here is a genuine piece of a model — periodic diffusion of the tracer `c` — written with nothing but the
ideas above. A periodic Laplacian by array shifts, then an explicit Euler step, in place:

In [27]:
laplacian(c, Δx) = (circshift(c, -1) .- 2 .* c .+ circshift(c, 1)) ./ Δx^2

function diffuse!(c, κ, Δx, Δt, steps)
    for _ in 1:steps
        c .+= κ * Δt .* laplacian(c, Δx)            # one fused broadcast per step
    end
    return c
end

κ = 0.5
Δt = 0.2 * Δx^2 / κ                                 # a stable explicit time step
c_diffused = diffuse!(copy(c), κ, Δx, Δt, 500)
@sprintf("peak amplitude: %.3f → %.3f after 500 steps", maximum(c), maximum(c_diffused))

"peak amplitude: 1.000 → 0.783 after 500 steps"

Notice what we did *not* do: no type annotations, no manual loops over indices, no separate "fast
version." This single function is already specialized by the compiler the first time it is called.

And the composability promise from above survives the field update: hand `diffuse!` a field of
*uncertain* values and the very same function — never given a method for `Measurements` — carries an
error bar at every grid point as it steps:

In [28]:
uncertain_field = c .± 0.05                 # each tracer value gains an uncertainty of ±0.05
diffuse!(uncertain_field, κ, Δx, Δt, 50)
uncertain_field[64]                         # the midpoint, with its propagated uncertainty

0.024 ± 0.013

## Seeing it: a first figure

`CairoMakie` draws figures from the same arrays. The high-wavenumber wave decays, as diffusion demands:

In [34]:
using CairoMakie

fig = Figure(size = (760, 380))
ax = Axis(fig[1, 1]; xlabel = "x", ylabel = "tracer", title = "explicit diffusion after 500 steps")
lines!(ax, x, c;          linewidth = 3, label = "initial")
lines!(ax, x, c_diffused; linewidth = 3, label = "diffused")
axislegend(ax)
save("julia_essentials_diffusion.png", fig)
nothing

![](julia_essentials_diffusion.png)

## You don't start from scratch: calling C and Fortran

Climate science sits on decades of Fortran and C — radiation schemes, ocean and atmosphere dynamical
cores, sea-ice rheologies. Julia does not ask you to rewrite them: `ccall` invokes a compiled C or Fortran
routine directly, with no wrapper and no copy. Here it reaches the system math library; naming a real
library — `ccall((:my_routine, "libradiation"), …)` — reaches a Fortran or C subroutine the same way:

In [ ]:
ccall(:cos, Float64, (Float64,), 1.0)             # call C's cos directly — no wrapper, native cost

And if your tooling lives in Python, the bridge runs both ways:
[`PythonCall.jl`](https://github.com/JuliaPy/PythonCall.jl) (or `PyCall.jl`) calls NumPy, xarray, or
Matplotlib from Julia, and drives a Julia model from a Python script — so a group can adopt Julia piece by
piece. The call reads like ordinary Julia:

In [ ]:
using PythonCall

np = pyimport("numpy")
temperatures = np.array([4.0, 3.2, 2.8, 1.5])     # a NumPy array, built from Julia
pyconvert(Float64, np.mean(temperatures))         # NumPy computes the mean, handed back to Julia

That is the whole toolkit: arrays and broadcasting for the fields, functions and dispatch for the
operations on them, a compiler that turns it into fast machine code, and — when you need them — the C,
Fortran, and Python libraries you already have. Every model script this week — ocean, atmosphere, or sea
ice — is built from exactly these pieces.

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*